# Qwen2.5-1.5B QLoRA + Random Forest — fixed Kaggle pipeline

This revision fixes two issues from the previous run:
1. Installs/verifies `bitsandbytes==0.50.2` before starting QLoRA (this version was present in your earlier successful T4 run).
2. Replaces the poorly balanced group split with a deterministic group-disjoint search that keeps whole scenario episodes together while matching ON/OFF proportions more closely.

**Kaggle requirement:** enable Internet in Notebook Settings if `bitsandbytes` is not already installed. Keep the GPU accelerator enabled. The script still uses your local Qwen checkpoint; it does not download model weights.

The full pipeline audits features with Random Forest, fine-tunes Qwen from the exact local checkpoint, compares models on one group-held-out test set, and runs boundary tests.

In [ ]:
# Preflight: install the bitsandbytes version that worked in the earlier T4 run.
import importlib.util
import subprocess
import sys

BNB_VERSION = '0.50.2'
if importlib.util.find_spec('bitsandbytes') is None:
    print(f'Installing bitsandbytes=={BNB_VERSION} ...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q', f'bitsandbytes=={BNB_VERSION}'])
else:
    print('bitsandbytes is already installed')

import torch
import bitsandbytes as bnb
print('bitsandbytes:', getattr(bnb, '__version__', 'unknown'))
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No CUDA GPU detected. Set Kaggle Notebook Settings > Accelerator > GPU.')


In [ ]:
""" 
RF feature audit + group-aware QLoRA fine-tuning for irrigation pump state.

Kaggle inputs:
  CSV:   /kaggle/input/datasets/nguyncngo/sensor-log/sensor_log.csv
  Model: /kaggle/input/models/qwen-lm/qwen2.5/transformers/1.5b/1

What this pipeline does:
  1) Recreates contiguous scenario episodes for GROUP SPLITTING ONLY.
  2) Audits whether labels follow the source threshold controller.
  3) Uses Random Forest + StratifiedGroupKFold to assess sensor features,
     including temperature, air humidity and light.
  4) Compares controller-only features with all six numerical sensors.
  5) Fine-tunes the supplied Qwen2.5-1.5B checkpoint with QLoRA using only
     the three controller inputs selected from source logic + RF audit.
  6) Evaluates Qwen, Random Forest and the deterministic controller on the
     same group-held-out test set.
  7) Saves models, metrics, feature importances, OOF predictions and tests.

Important:
  - The episode/scenario column is NEVER used as a model feature.
  - The provided dataset is synthetic and its labels match the source rule.
  - A Random Forest finding of low rain importance does NOT disprove the
    source rain override: the dataset has no records where rain > 60%,
    soil < 30% and water >= 20% simultaneously. The rain override therefore
    is not independently isolated by the observations.
  - For actual pump actuation, retain a deterministic safety controller and
    hardware interlocks; do not connect LLM output directly to a pump relay.
"""

import os
import re
import json
import random
import inspect
import shutil
import warnings
import subprocess
import sys
import importlib
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed,
    default_data_collator,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from torch.utils.data import Dataset


# ============================================================================
# 1. CONFIGURATION
# ============================================================================
DATA_PATH = "/kaggle/input/datasets/nguyncngo/sensor-log/sensor_log.csv"
BASE_MODEL_PATH = "/kaggle/input/models/qwen-lm/qwen2.5/transformers/1.5b/1"

WORK_DIR = Path("/kaggle/working/pump_rf_qwen")
QWEN_OUTPUT_DIR = WORK_DIR / "qwen_pump_lora"
WORK_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
MAX_SEQ_LENGTH = 192

# RF settings. Kept modest because this is a 354-row dataset.
RF_N_ESTIMATORS = 200
RF_MIN_SAMPLES_LEAF = 2
RF_N_JOBS = -1
RF_PERMUTATION_REPEATS = 10

# QLoRA settings. Early stopping chooses the best group-validation checkpoint.
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4
TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
EARLY_STOPPING_PATIENCE = 2
BITSANDBYTES_VERSION = "0.50.2"  # Previously verified in this Kaggle/T4 workflow.
GROUP_SPLIT_SEARCH_TRIALS = 1000

TARGET_CANDIDATES = ["pump_status", "pump_state"]
CONTROLLER_FEATURES = [
    "soil_moisture_%",
    "water_level_%",
    "rain_level_%",
]
ENVIRONMENT_FEATURES = [
    "temperature_C",
    "humidity_%",
    "light_intensity_%",
]
ALL_NUMERIC_FEATURES = CONTROLLER_FEATURES + ENVIRONMENT_FEATURES


# ============================================================================
# 2. REPRODUCIBILITY, CSV AND PATH HELPERS
# ============================================================================
def seed_everything(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)


def check_paths():
    if not Path(DATA_PATH).is_file():
        raise FileNotFoundError(f"CSV not found: {DATA_PATH}. Attach the Kaggle dataset.")
    if not Path(BASE_MODEL_PATH).is_dir():
        raise FileNotFoundError(
            f"Qwen model directory not found: {BASE_MODEL_PATH}. Attach the Kaggle model input."
        )
    if not (Path(BASE_MODEL_PATH) / "config.json").is_file():
        raise FileNotFoundError(f"config.json is missing under {BASE_MODEL_PATH}")


def read_csv_robust(path):
    last_error = None
    for enc in ("utf-8-sig", "utf-8", "cp1252", "latin1"):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise RuntimeError(f"Cannot decode {path}: {last_error}")


def normalize_scenario_text(series):
    """Fix common mojibake variants for grouping/audit; never used as a feature."""
    s = series.fillna("UNKNOWN").astype(str)
    s = s.str.replace("â€”", "—", regex=False)
    s = s.str.replace("â€“", "–", regex=False)
    s = s.str.replace("â€™", "’", regex=False)
    return s.str.strip()


def add_episode_groups(df):
    """Create contiguous scenario-episode IDs for group splits only.

    A new group begins when the scenario changes or reading_no resets.
    This avoids scattering adjacent readings from the same generated episode
    across train/validation/test. The generated group ID is never model input.
    """
    out = df.copy().reset_index(drop=True)
    if "scenario" not in out.columns:
        raise KeyError("Column 'scenario' is required to construct episode groups.")

    scenario = normalize_scenario_text(out["scenario"])
    new_group = scenario.ne(scenario.shift())

    if "reading_no" in out.columns:
        reading_no = pd.to_numeric(out["reading_no"], errors="coerce")
        reset_event = reading_no.diff().fillna(1).lt(0)
        new_group = new_group | reset_event

    out["scenario_group_only"] = scenario
    out["episode_group"] = new_group.cumsum().astype(int)
    return out


def load_and_prepare_data():
    df = read_csv_robust(DATA_PATH)
    df.columns = [str(c).strip() for c in df.columns]

    target_col = next((c for c in TARGET_CANDIDATES if c in df.columns), None)
    if target_col is None:
        raise KeyError(f"Expected one of {TARGET_CANDIDATES}; found {df.columns.tolist()}")

    required = ALL_NUMERIC_FEATURES + ["scenario"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    if "reading_no" in df.columns:
        df["reading_no"] = pd.to_numeric(df["reading_no"], errors="coerce")

    for col in ALL_NUMERIC_FEATURES:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df[target_col] = df[target_col].astype(str).str.strip().str.upper()
    df = df[df[target_col].isin(["ON", "OFF"])].copy()
    df = df.dropna(subset=ALL_NUMERIC_FEATURES + [target_col, "scenario"]).reset_index(drop=True)
    df = df.rename(columns={target_col: "pump_status"})
    df = add_episode_groups(df)

    if len(df) < 30:
        raise ValueError(f"Only {len(df)} usable rows remain; expected at least 30.")
    if df["pump_status"].nunique() != 2:
        raise ValueError("Both ON and OFF labels must be present.")

    return df


# ============================================================================
# 3. SOURCE CONTROLLER FUNCTION AND DATA AUDIT
# ============================================================================
def source_rule_predict(frame):
    """Replicate the source project's pump logic for audit and safety baseline."""
    soil = np.asarray(frame["soil_moisture_%"], dtype=float)
    water = np.asarray(frame["water_level_%"], dtype=float)
    rain = np.asarray(frame["rain_level_%"], dtype=float)

    # Source decision logic:
    # rain > 60% -> OFF; water < 20% -> OFF; soil < 30% -> ON; otherwise OFF.
    return np.where(
        (rain <= 60.0) & (water >= 20.0) & (soil < 30.0),
        "ON",
        "OFF",
    )


def rule_without_rain(frame):
    soil = np.asarray(frame["soil_moisture_%"], dtype=float)
    water = np.asarray(frame["water_level_%"], dtype=float)
    return np.where((water >= 20.0) & (soil < 30.0), "ON", "OFF")


def print_data_audit(df):
    y = df["pump_status"].to_numpy()
    rule_pred = source_rule_predict(df)
    agreement = accuracy_score(y, rule_pred)

    print("\n" + "=" * 78)
    print("DATA AUDIT")
    print("=" * 78)
    print("Rows:", len(df))
    print("Target distribution:\n", df["pump_status"].value_counts().to_string())
    print("Scenario episodes:", df["episode_group"].nunique())
    print("Scenario group sizes:\n", df.groupby("episode_group").size().value_counts().sort_index().to_string())
    print(f"Source-rule agreement: {agreement:.4%}")
    print("Qwen/RF controller features:", CONTROLLER_FEATURES)
    print("RF-only candidate environmental features:", ENVIRONMENT_FEATURES)
    print("Excluded from model features: timestamp, reading_no, scenario, alerts, episode_group")

    print("\nSensor ranges:")
    print(df[ALL_NUMERIC_FEATURES].agg(["min", "median", "max"]).T.to_string())

    rain_high = df["rain_level_%"] > 60.0
    otherwise_on = (df["soil_moisture_%"] < 30.0) & (df["water_level_%"] >= 20.0)
    isolated_rain_override_cases = int((rain_high & otherwise_on).sum())
    print("\nRain > 60% records:", int(rain_high.sum()))
    print("Rain > 60% AND soil < 30% AND water >= 20%:", isolated_rain_override_cases)
    if isolated_rain_override_cases == 0:
        print(
            "NOTE: The rain threshold exists in the source rule, but this dataset "
            "does not isolate a case where rain is the only reason to turn the pump OFF."
        )

    rule_ablation = {
        "full_source_rule": source_rule_predict(df),
        "rule_without_rain_condition": rule_without_rain(df),
        "rule_without_water_condition": np.where(df["soil_moisture_%"] < 30.0, "ON", "OFF"),
        "rule_without_soil_condition": np.where(
            (df["rain_level_%"] <= 60.0) & (df["water_level_%"] >= 20.0), "ON", "OFF"
        ),
    }
    rows = []
    for name, pred in rule_ablation.items():
        rows.append({"rule_variant": name, "accuracy": accuracy_score(y, pred)})
    rule_df = pd.DataFrame(rows)
    rule_df.to_csv(WORK_DIR / "source_rule_ablation.csv", index=False)
    print("\nSource rule ablation:\n", rule_df.to_string(index=False))

    corr = df[ALL_NUMERIC_FEATURES].corrwith((df["pump_status"] == "ON").astype(int))
    corr.rename("pearson_corr_with_ON").to_csv(WORK_DIR / "sensor_target_correlations.csv")
    print("\nPearson correlation with Pump ON (descriptive only):\n", corr.round(4).to_string())

    if agreement == 1.0:
        print(
            "\nWARNING: Every label is generated by the deterministic rule. "
            "The task is rule imitation, not evidence of a new physical relationship."
        )


# ============================================================================
# 4. RANDOM FOREST GROUP-CV FEATURE AUDIT
# ============================================================================
def build_rf(seed=RANDOM_SEED):
    return RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS,
        min_samples_leaf=RF_MIN_SAMPLES_LEAF,
        max_features=None,
        class_weight="balanced",
        random_state=seed,
        n_jobs=RF_N_JOBS,
    )


def metric_row(y_true, y_pred, name, n_features):
    return {
        "model": name,
        "n_features": n_features,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_ON": precision_score(y_true, y_pred, pos_label="ON", zero_division=0),
        "recall_ON": recall_score(y_true, y_pred, pos_label="ON", zero_division=0),
        "f1_ON": f1_score(y_true, y_pred, pos_label="ON", zero_division=0),
    }


def run_rf_group_cv(df):
    y = df["pump_status"].to_numpy()
    groups = df["episode_group"].to_numpy()
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

    ablation_sets = {
        "controller_3": CONTROLLER_FEATURES,
        "all_sensor_6": ALL_NUMERIC_FEATURES,
        "all_6_without_temperature": [c for c in ALL_NUMERIC_FEATURES if c != "temperature_C"],
        "all_6_without_humidity": [c for c in ALL_NUMERIC_FEATURES if c != "humidity_%"],
        "all_6_without_light": [c for c in ALL_NUMERIC_FEATURES if c != "light_intensity_%"],
        "all_6_without_rain": [c for c in ALL_NUMERIC_FEATURES if c != "rain_level_%"],
        "controller_without_rain": ["soil_moisture_%", "water_level_%"],
        "controller_without_water": ["soil_moisture_%", "rain_level_%"],
        "controller_without_soil": ["water_level_%", "rain_level_%"],
    }

    all_oof = {}
    result_rows = []
    for name, features in ablation_sets.items():
        X = df[features].copy()
        estimator = build_rf()
        pred = cross_val_predict(
            estimator,
            X,
            y,
            cv=cv,
            groups=groups,
            method="predict",
            n_jobs=1,
        )
        all_oof[name] = pred
        result_rows.append(metric_row(y, pred, name, len(features)))
        print(
            f"RF GroupCV {name:32s} | "
            f"Accuracy={accuracy_score(y, pred):.4f} | "
            f"BalancedAcc={balanced_accuracy_score(y, pred):.4f} | "
            f"F1_ON={f1_score(y, pred, pos_label='ON'):.4f}"
        )

    ablation_df = pd.DataFrame(result_rows).sort_values(
        ["balanced_accuracy", "accuracy"], ascending=False
    )
    ablation_df.to_csv(WORK_DIR / "rf_groupcv_ablation.csv", index=False)

    # Detailed grouped out-of-fold predictions for the two main feature sets.
    oof_df = df[["timestamp", "reading_no", "scenario_group_only", "episode_group", "pump_status"]].copy()
    oof_df["rf_controller3_oof"] = all_oof["controller_3"]
    oof_df["rf_all6_oof"] = all_oof["all_sensor_6"]
    oof_df.to_csv(WORK_DIR / "rf_groupcv_oof_predictions.csv", index=False)

    # Group-held-out permutation importance: evaluate each fold only on groups
    # that were not used to fit that fold's RF.
    mdi_by_fold = []
    permutation_values = {c: [] for c in ALL_NUMERIC_FEATURES}
    fold_rows = []
    X_all = df[ALL_NUMERIC_FEATURES]
    for fold, (train_idx, test_idx) in enumerate(cv.split(X_all, y, groups), start=1):
        model = build_rf(seed=RANDOM_SEED + fold)
        model.fit(X_all.iloc[train_idx], y[train_idx])
        pred = model.predict(X_all.iloc[test_idx])
        fold_rows.append({
            "fold": fold,
            "n_test": len(test_idx),
            "n_test_groups": len(np.unique(groups[test_idx])),
            "accuracy": accuracy_score(y[test_idx], pred),
            "balanced_accuracy": balanced_accuracy_score(y[test_idx], pred),
        })
        mdi_by_fold.append(model.feature_importances_)
        perm = permutation_importance(
            model,
            X_all.iloc[test_idx],
            y[test_idx],
            n_repeats=RF_PERMUTATION_REPEATS,
            random_state=RANDOM_SEED + fold,
            scoring="balanced_accuracy",
            n_jobs=1,
        )
        for feature, values in zip(ALL_NUMERIC_FEATURES, perm.importances_mean):
            permutation_values[feature].append(float(values))

    importance_rows = []
    mdi_mean = np.mean(mdi_by_fold, axis=0)
    mdi_std = np.std(mdi_by_fold, axis=0)
    for i, feature in enumerate(ALL_NUMERIC_FEATURES):
        pvals = np.asarray(permutation_values[feature], dtype=float)
        importance_rows.append({
            "feature": feature,
            "mean_impurity_importance": float(mdi_mean[i]),
            "std_impurity_importance": float(mdi_std[i]),
            "mean_grouped_permutation_drop_balanced_accuracy": float(pvals.mean()),
            "std_grouped_permutation_drop_balanced_accuracy": float(pvals.std()),
        })
    importance_df = pd.DataFrame(importance_rows).sort_values(
        "mean_grouped_permutation_drop_balanced_accuracy", ascending=False
    )
    importance_df.to_csv(WORK_DIR / "rf_grouped_feature_importance.csv", index=False)
    pd.DataFrame(fold_rows).to_csv(WORK_DIR / "rf_groupcv_fold_metrics.csv", index=False)

    print("\nRF ablation results:\n", ablation_df.to_string(index=False))
    print("\nGrouped permutation feature importance:\n", importance_df.to_string(index=False))

    # Feature-selection decision: stay with the three source-controller inputs.
    # The extra environmental sensors are diagnostic candidates, not controller
    # inputs. We do not automatically add a feature just because it correlates.
    best_core = ablation_df.loc[ablation_df["model"] == "controller_3"].iloc[0]
    best_all = ablation_df.loc[ablation_df["model"] == "all_sensor_6"].iloc[0]
    if best_core["balanced_accuracy"] >= best_all["balanced_accuracy"]:
        selected_features = CONTROLLER_FEATURES.copy()
    else:
        # Still require source logic support before adding environmental features.
        selected_features = CONTROLLER_FEATURES.copy()

    print("\nSelected Qwen features:", selected_features)
    print("Reason: the source controller uses these inputs; extra environment features did not improve grouped CV.")

    # Fit a final RF artifact on all rows for optional inference only.
    final_rf = build_rf()
    final_rf.fit(df[selected_features], y)
    joblib.dump(
        {"model": final_rf, "features": selected_features, "target_labels": ["OFF", "ON"]},
        WORK_DIR / "rf_pump_final.joblib",
    )

    with open(WORK_DIR / "rf_audit_summary.json", "w", encoding="utf-8") as f:
        json.dump({
            "selected_qwen_features": selected_features,
            "group_count": int(df["episode_group"].nunique()),
            "group_cv_ablation": ablation_df.to_dict(orient="records"),
            "feature_importance": importance_df.to_dict(orient="records"),
        }, f, indent=2)

    return selected_features, ablation_df, importance_df, all_oof


# ============================================================================
# 5. GROUP-AWARE TRAIN / VALIDATION / TEST SPLIT FOR QWEN
# ============================================================================
def make_group_splits(df):
    """Create a deterministic group-disjoint split with balanced class ratios.

    The previous nested StratifiedGroupKFold split kept groups disjoint but,
    for this small dataset, produced a validation set with 29 ON / 27 OFF and
    a test set with only 7 ON / 59 OFF. This search assigns whole episodes and
    selects the partition whose sample counts and ON/OFF ratios are closest
    to 64% / 16% / 20% of the full dataset.

    Labels are used only for stratifying the split, never as model features.
    """
    group_col = "episode_group"
    if group_col not in df.columns:
        raise KeyError("episode_group is required for group-aware splitting")

    grouped = df.groupby(group_col)["pump_status"].agg(
        n="size",
        on_count=lambda s: int((s == "ON").sum()),
    )
    group_ids = grouped.index.to_numpy()
    group_stats = grouped[["n", "on_count"]].to_numpy(dtype=float)
    n_groups = len(group_ids)
    if n_groups < 10:
        raise ValueError(f"Need at least 10 independent episode groups; found {n_groups}.")

    # Target proportions: 64% train, 16% validation, 20% test.
    fractions = np.array([0.64, 0.16, 0.20], dtype=float)
    target_sizes = fractions * len(df)
    total_on = int((df["pump_status"] == "ON").sum())
    global_on_rate = total_on / len(df)
    target_on = target_sizes * global_on_rate

    # Keep whole groups. Rounded group counts sum exactly to n_groups.
    n_train_groups = int(round(0.64 * n_groups))
    n_val_groups = int(round(0.16 * n_groups))
    n_test_groups = n_groups - n_train_groups - n_val_groups
    if min(n_train_groups, n_val_groups, n_test_groups) < 1:
        raise ValueError("Not enough groups to create train/validation/test partitions.")

    rng = np.random.default_rng(RANDOM_SEED)
    best = None
    for trial in range(GROUP_SPLIT_SEARCH_TRIALS):
        perm = rng.permutation(n_groups)
        group_indices = [
            perm[:n_train_groups],
            perm[n_train_groups:n_train_groups + n_val_groups],
            perm[n_train_groups + n_val_groups:],
        ]
        sizes = np.array([group_stats[idx, 0].sum() for idx in group_indices])
        positives = np.array([group_stats[idx, 1].sum() for idx in group_indices])
        if np.any(sizes == 0):
            continue

        on_rates = positives / sizes
        size_error = np.abs(sizes - target_sizes) / np.maximum(target_sizes, 1.0)
        positive_error = np.abs(positives - target_on) / np.maximum(target_on, 1.0)
        rate_error = np.abs(on_rates - global_on_rate)

        # Ensure validation and test contain a usable number of both classes.
        min_positive = max(3.0, 0.65 * target_on[1])
        min_negative = max(3.0, 0.65 * (target_sizes[1] - target_on[1]))
        low_class_penalty = 0.0
        for split_i in (1, 2):
            neg = sizes[split_i] - positives[split_i]
            low_class_penalty += max(0.0, min_positive - positives[split_i]) / min_positive
            low_class_penalty += max(0.0, min_negative - neg) / min_negative

        objective = (
            size_error.sum()
            + 1.25 * positive_error.sum()
            + 0.50 * rate_error.sum()
            + 0.70 * low_class_penalty
        )
        if best is None or objective < best[0]:
            best = (objective, group_indices, sizes, positives, on_rates, trial)

    if best is None:
        raise RuntimeError("Could not construct a valid group-aware split.")

    _, group_indices, sizes, positives, on_rates, best_trial = best
    selected_groups = [set(group_ids[idx].tolist()) for idx in group_indices]
    partitions = [df[df[group_col].isin(gset)].copy() for gset in selected_groups]
    train_df, val_df, test_df = [part.reset_index(drop=True) for part in partitions]

    group_sets = [set(part[group_col].unique()) for part in (train_df, val_df, test_df)]
    assert group_sets[0].isdisjoint(group_sets[1])
    assert group_sets[0].isdisjoint(group_sets[2])
    assert group_sets[1].isdisjoint(group_sets[2])

    print("\nBALANCED GROUP-AWARE QWEN SPLIT")
    print(f"Split search trials: {GROUP_SPLIT_SEARCH_TRIALS}; selected trial: {best_trial}")
    for name, part in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
        counts = part["pump_status"].value_counts().to_dict()
        print(
            f"{name:10s}: n={len(part):3d}, groups={part[group_col].nunique():2d}, "
            f"labels={counts}, ON_rate={(part['pump_status'] == 'ON').mean():.3f}"
        )
    print("Group overlap: NONE")

    # Persist split membership for exact reproducibility and post-run audit.
    split_map = pd.concat([
        train_df.assign(split="train"),
        val_df.assign(split="validation"),
        test_df.assign(split="test"),
    ], ignore_index=True)[[group_col, "split"]].drop_duplicates()
    split_map.to_csv(WORK_DIR / "qwen_group_split_membership.csv", index=False)
    return train_df, val_df, test_df


# ============================================================================
# 6. QWEN PROMPTS AND TOKENIZED DATASET
# ============================================================================
def make_prompt(row):
    return (
        "### Task\n"
        "Classify the irrigation controller pump state from the current sensor readings.\n"
        "Return exactly one word: ON or OFF. Do not add an explanation.\n\n"
        "### Current sensor readings\n"
        f"Soil moisture: {float(row['soil_moisture_%']):.1f}%\n"
        f"Water tank level: {float(row['water_level_%']):.1f}%\n"
        f"Rain level: {float(row['rain_level_%']):.1f}%\n\n"
        "### Pump state\n"
    )


class PumpTextDataset(Dataset):
    def __init__(self, frame, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.features = frame[CONTROLLER_FEATURES].copy().reset_index(drop=True)
        self.targets = frame["pump_status"].astype(str).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.examples = [self._encode(i) for i in range(len(self.features))]

    def _encode(self, idx):
        row = self.features.iloc[idx]
        answer = self.targets[idx]
        prompt = make_prompt(row)
        full_text = prompt + answer + (self.tokenizer.eos_token or "")

        prompt_ids = self.tokenizer(prompt, add_special_tokens=False)["input_ids"]
        full_ids = self.tokenizer(full_text, add_special_tokens=False)["input_ids"]

        if full_ids[:len(prompt_ids)] != prompt_ids:
            raise RuntimeError("Tokenizer boundary mismatch; prompt masking would be incorrect.")
        if len(full_ids) > self.max_length:
            full_ids = full_ids[:self.max_length]
        prompt_len = min(len(prompt_ids), len(full_ids))

        labels = list(full_ids)
        labels[:prompt_len] = [-100] * prompt_len

        pad_id = self.tokenizer.pad_token_id
        pad_len = self.max_length - len(full_ids)
        input_ids = list(full_ids) + [pad_id] * pad_len
        attention_mask = [1] * len(full_ids) + [0] * pad_len
        labels += [-100] * pad_len

        if not any(x != -100 for x in labels):
            raise RuntimeError("All target tokens were masked; increase MAX_SEQ_LENGTH.")

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


# ============================================================================
# 7. LOAD BASE QWEN CHECKPOINT + QLORA
# ============================================================================
def ensure_bitsandbytes():
    """Import bitsandbytes; install the known-working Kaggle version if absent."""
    try:
        return importlib.import_module("bitsandbytes")
    except ModuleNotFoundError as exc:
        if exc.name != "bitsandbytes":
            raise
        print(
            f"bitsandbytes is missing. Installing bitsandbytes=={BITSANDBYTES_VERSION}..."
        )
        try:
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", "--no-cache-dir", "-q",
                f"bitsandbytes=={BITSANDBYTES_VERSION}",
            ])
        except subprocess.CalledProcessError as install_error:
            raise RuntimeError(
                "Automatic bitsandbytes installation failed. In Kaggle, enable "
                "Internet under Notebook Settings, run `%pip install -q "
                f"bitsandbytes=={BITSANDBYTES_VERSION}` in a separate cell, restart "
                "the session if prompted, and rerun the notebook."
            ) from install_error
        importlib.invalidate_caches()
        try:
            return importlib.import_module("bitsandbytes")
        except Exception as import_error:
            raise RuntimeError(
                "bitsandbytes was installed but could not be imported. Check the "
                "PyTorch/CUDA compatibility and the Kaggle GPU runtime."
            ) from import_error


def load_qwen_qlora():
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA GPU not found. In Kaggle select Notebook Settings > Accelerator > GPU."
        )

    bnb = ensure_bitsandbytes()
    print("bitsandbytes:", getattr(bnb, "__version__", "version not exposed"))
    print("CUDA:", torch.cuda.is_available())
    print("GPU:", torch.cuda.get_device_name(0))

    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        BASE_MODEL_PATH,
        local_files_only=True,
        trust_remote_code=False,
        use_fast=True,
    )
    if tokenizer.eos_token is None:
        raise ValueError("Qwen tokenizer has no EOS token.")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # Support Transformers versions that use either `dtype` or `torch_dtype`.
    load_params = inspect.signature(AutoModelForCausalLM.from_pretrained).parameters
    dtype_key = "dtype" if "dtype" in load_params else "torch_dtype"
    model_kwargs = {
        "quantization_config": quant_config,
        "device_map": "auto",
        "local_files_only": True,
        "trust_remote_code": False,
    }
    model_kwargs[dtype_key] = compute_dtype

    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_PATH, **model_kwargs)
    model.config.use_cache = False
    model.config.pad_token_id = tokenizer.pad_token_id
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model, tokenizer, compute_dtype


# ============================================================================
# 8. TRAIN WITH GROUP VALIDATION + EARLY STOPPING
# ============================================================================
def make_training_arguments(compute_dtype, train_size):
    """Build TrainingArguments compatible with the installed Transformers API.

    Some Kaggle images expose a TrainingArguments signature that does not
    accept `warmup_ratio`. Use the broadly supported `warmup_steps` instead,
    and filter optional kwargs against the installed constructor signature.
    """
    params = inspect.signature(TrainingArguments.__init__).parameters

    if "eval_strategy" in params:
        eval_key = "eval_strategy"
    elif "evaluation_strategy" in params:
        eval_key = "evaluation_strategy"
    else:
        raise RuntimeError(
            "Installed TrainingArguments has neither eval_strategy nor "
            "evaluation_strategy; upgrade Transformers to a compatible release."
        )

    # Approximate optimizer update steps for a single-GPU Kaggle run.
    # This dataset uses one T4, so world_size=1.
    micro_batches_per_epoch = math.ceil(train_size / TRAIN_BATCH_SIZE)
    optimizer_steps_per_epoch = math.ceil(
        micro_batches_per_epoch / GRADIENT_ACCUMULATION_STEPS
    )
    total_optimizer_steps = max(1, math.ceil(optimizer_steps_per_epoch * NUM_EPOCHS))
    warmup_steps = max(0, math.ceil(total_optimizer_steps * 0.05))

    kwargs = {
        "output_dir": str(QWEN_OUTPUT_DIR),
        "num_train_epochs": NUM_EPOCHS,
        "per_device_train_batch_size": TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size": EVAL_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": 0.01,
        "warmup_steps": warmup_steps,
        "lr_scheduler_type": "cosine",
        "optim": "paged_adamw_8bit",
        "logging_steps": 5,
        "save_strategy": "epoch",
        "load_best_model_at_end": True,
        "metric_for_best_model": "eval_loss",
        "greater_is_better": False,
        "save_total_limit": 2,
        "fp16": compute_dtype == torch.float16,
        "bf16": compute_dtype == torch.bfloat16,
        "gradient_checkpointing": True,
        "max_grad_norm": 0.3,
        "report_to": "none",
        "seed": RANDOM_SEED,
        "data_seed": RANDOM_SEED,
        "remove_unused_columns": False,
        eval_key: "epoch",
    }

    # Avoid `unexpected keyword argument` across Kaggle's varying
    # Transformers builds. Required training/evaluation fields are checked.
    supported_kwargs = {key: value for key, value in kwargs.items() if key in params}
    dropped = sorted(set(kwargs) - set(supported_kwargs))
    required = [
        "output_dir",
        "num_train_epochs",
        "per_device_train_batch_size",
        "gradient_accumulation_steps",
        "learning_rate",
        eval_key,
    ]
    missing_required = [key for key in required if key not in supported_kwargs]
    if missing_required:
        raise RuntimeError(
            "Installed TrainingArguments is missing required parameters: "
            f"{missing_required}. Transformers version: "
            f"{getattr(__import__('transformers'), '__version__', 'unknown')}"
        )

    print(f"TrainingArguments compatibility: {len(supported_kwargs)} supported kwargs")
    print(f"Warmup steps: {warmup_steps} / {total_optimizer_steps} estimated optimizer steps")
    if dropped:
        print("Optional unsupported TrainingArguments fields skipped:", dropped)

    return TrainingArguments(**supported_kwargs)


def train_qwen(train_df, val_df):
    model, tokenizer, compute_dtype = load_qwen_qlora()
    train_ds = PumpTextDataset(train_df, tokenizer)
    val_ds = PumpTextDataset(val_df, tokenizer)
    args = make_training_arguments(compute_dtype, train_size=len(train_ds))

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=default_data_collator,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=EARLY_STOPPING_PATIENCE,
                early_stopping_threshold=1e-6,
            )
        ],
    )

    print("\nStarting QLoRA fine-tuning from:", BASE_MODEL_PATH)
    train_output = trainer.train()
    print("Training metrics:", train_output.metrics)
    best_eval = trainer.evaluate()
    print("Best-checkpoint validation:", best_eval)

    trainer.save_model(str(QWEN_OUTPUT_DIR))
    tokenizer.save_pretrained(str(QWEN_OUTPUT_DIR))
    trainer.save_state()

    with open(WORK_DIR / "qwen_training_summary.json", "w", encoding="utf-8") as f:
        json.dump({
            "base_model": BASE_MODEL_PATH,
            "best_checkpoint": trainer.state.best_model_checkpoint,
            "best_metric": trainer.state.best_metric,
            "train_metrics": train_output.metrics,
            "best_validation_metrics": best_eval,
            "epochs_configured": NUM_EPOCHS,
            "epochs_completed": trainer.state.epoch,
        }, f, indent=2, default=str)

    return trainer.model, tokenizer, trainer


# ============================================================================
# 9. INFERENCE + EVALUATION
# ============================================================================
def parse_pump_label(text):
    match = re.search(r"\b(ON|OFF)\b", str(text).upper())
    return match.group(1) if match else "UNKNOWN"


@torch.inference_mode()
def predict_qwen_one(model, tokenizer, sensor_values):
    prompt = make_prompt(sensor_values)
    encoded = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    input_device = model.get_input_embeddings().weight.device
    encoded = {k: v.to(input_device) for k, v in encoded.items()}
    output = model.generate(
        **encoded,
        max_new_tokens=5,
        do_sample=False,
        num_beams=1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )
    new_tokens = output[0, encoded["input_ids"].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return parse_pump_label(raw), raw


def calculate_metrics(y_true, y_pred, title):
    y_true = np.asarray(y_true, dtype=str)
    y_pred = np.asarray(y_pred, dtype=str)
    accuracy = float(np.mean(y_true == y_pred))
    valid = np.isin(y_pred, ["ON", "OFF"])
    result = {
        "accuracy": accuracy,
        "parse_success_rate": float(valid.mean()),
        "balanced_accuracy": None,
        "precision_ON": None,
        "recall_ON": None,
        "f1_ON": None,
        "confusion_matrix_labels": ["OFF", "ON", "UNKNOWN"],
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=["OFF", "ON", "UNKNOWN"]).tolist(),
    }
    if valid.any():
        yt = y_true[valid]
        yp = y_pred[valid]
        result.update({
            "balanced_accuracy": float(balanced_accuracy_score(yt, yp)),
            "precision_ON": float(precision_score(yt, yp, pos_label="ON", zero_division=0)),
            "recall_ON": float(recall_score(yt, yp, pos_label="ON", zero_division=0)),
            "f1_ON": float(f1_score(yt, yp, pos_label="ON", zero_division=0)),
        })
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)
    print(f"Parse success:    {result['parse_success_rate']:.2%}")
    print(f"Accuracy:         {result['accuracy']:.4f}")
    print(f"Balanced accuracy: {result['balanced_accuracy'] if result['balanced_accuracy'] is not None else 'N/A'}")
    print(f"Precision ON:      {result['precision_ON'] if result['precision_ON'] is not None else 'N/A'}")
    print(f"Recall ON:         {result['recall_ON'] if result['recall_ON'] is not None else 'N/A'}")
    print(f"F1 ON:             {result['f1_ON'] if result['f1_ON'] is not None else 'N/A'}")
    print(pd.DataFrame(
        result["confusion_matrix"],
        index=["true_OFF", "true_ON", "true_UNKNOWN"],
        columns=["pred_OFF", "pred_ON", "pred_UNKNOWN"],
    ).to_string())
    return result


def evaluate_group_test(model, tokenizer, train_df, val_df, test_df, rf_features):
    qwen_predictions, raw_outputs = [], []
    for i, (_, row) in enumerate(test_df.iterrows(), start=1):
        pred, raw = predict_qwen_one(model, tokenizer, row)
        qwen_predictions.append(pred)
        raw_outputs.append(raw)
        if i % 10 == 0 or i == len(test_df):
            print(f"Qwen generated {i}/{len(test_df)} test predictions")

    # RF is fitted only on train+validation groups; final score is on held-out groups.
    rf_train = pd.concat([train_df, val_df], ignore_index=True)
    rf_test_model = build_rf()
    rf_test_model.fit(rf_train[rf_features], rf_train["pump_status"])
    rf_predictions = rf_test_model.predict(test_df[rf_features])

    rule_predictions = source_rule_predict(test_df)

    qwen_metrics = calculate_metrics(test_df["pump_status"], qwen_predictions, "QWEN GROUP-HELD-OUT TEST")
    rf_metrics = calculate_metrics(test_df["pump_status"], rf_predictions, "RANDOM FOREST GROUP-HELD-OUT TEST")
    rule_metrics = calculate_metrics(test_df["pump_status"], rule_predictions, "SOURCE RULE GROUP-HELD-OUT TEST")

    result = test_df.copy().reset_index(drop=True)
    result["qwen_prediction"] = qwen_predictions
    result["qwen_raw_output"] = raw_outputs
    result["rf_prediction"] = rf_predictions
    result["source_rule_prediction"] = rule_predictions
    result.to_csv(WORK_DIR / "qwen_rf_group_test_predictions.csv", index=False)

    with open(WORK_DIR / "qwen_rf_group_test_metrics.json", "w", encoding="utf-8") as f:
        json.dump({
            "qwen": qwen_metrics,
            "random_forest": rf_metrics,
            "source_rule": rule_metrics,
            "test_rows": len(test_df),
            "test_groups": int(test_df["episode_group"].nunique()),
        }, f, indent=2)

    return result


# ============================================================================
# 10. BOUNDARY TESTS AND DEPLOYMENT-FACING FUNCTION
# ============================================================================
def source_controller_function(soil_moisture, water_level, rain_level):
    """Exact deterministic controller reference from the source thresholds."""
    if rain_level > 60.0:
        return "OFF"
    if water_level < 20.0:
        return "OFF"
    if soil_moisture < 30.0:
        return "ON"
    return "OFF"


def predict_pump_state_qwen(model, tokenizer, soil_moisture, water_level, rain_level):
    """Qwen-only state prediction; temperature/humidity/light are not inputs."""
    sample = {
        "soil_moisture_%": float(soil_moisture),
        "water_level_%": float(water_level),
        "rain_level_%": float(rain_level),
    }
    prediction, raw = predict_qwen_one(model, tokenizer, sample)
    return {"prediction": prediction, "raw_output": raw}


def predict_pump_state_safeguarded(model, tokenizer, soil_moisture, water_level, rain_level):
    """Return Qwen diagnostics plus source-rule final decision.

    The deterministic source rule is the final authority. This is intentional:
    an LLM must not directly energize a physical pump relay.
    """
    qwen_result = predict_pump_state_qwen(
        model, tokenizer, soil_moisture, water_level, rain_level
    )
    controller = source_controller_function(soil_moisture, water_level, rain_level)
    return {
        "qwen_prediction": qwen_result["prediction"],
        "qwen_raw_output": qwen_result["raw_output"],
        "controller_reference": controller,
        "final_safe_state": controller,
        "agreement": qwen_result["prediction"] == controller,
    }


def run_boundary_tests(model, tokenizer):
    cases = [
        (29.9, 70.0, 10.0, "ON", "soil just below 30"),
        (30.0, 70.0, 10.0, "OFF", "soil exactly 30"),
        (30.1, 70.0, 10.0, "OFF", "soil just above 30"),
        (20.0, 19.9, 10.0, "OFF", "water just below 20"),
        (20.0, 20.0, 10.0, "ON", "water exactly 20"),
        (20.0, 70.0, 60.0, "ON", "rain exactly 60"),
        (20.0, 70.0, 60.1, "OFF", "rain just above 60"),
        (20.0, 70.0, 90.0, "OFF", "heavy rain override"),
    ]
    rows = []
    for soil, water, rain, expected, note in cases:
        qwen_pred, raw = predict_qwen_one(model, tokenizer, {
            "soil_moisture_%": soil,
            "water_level_%": water,
            "rain_level_%": rain,
        })
        rule = source_controller_function(soil, water, rain)
        rows.append({
            "soil_moisture_%": soil,
            "water_level_%": water,
            "rain_level_%": rain,
            "expected_source_rule": expected,
            "source_rule_prediction": rule,
            "qwen_prediction": qwen_pred,
            "qwen_raw_output": raw,
            "qwen_correct": qwen_pred == expected,
            "case": note,
        })
    boundary_df = pd.DataFrame(rows)
    boundary_df.to_csv(WORK_DIR / "qwen_boundary_tests.csv", index=False)
    print("\nBoundary tests:\n", boundary_df.to_string(index=False))
    return boundary_df


# ============================================================================
# 11. MAIN
# ============================================================================
def main():
    seed_everything(RANDOM_SEED)
    check_paths()
    df = load_and_prepare_data()
    print_data_audit(df)

    selected_features, rf_ablation, rf_importance, rf_oof = run_rf_group_cv(df)
    train_df, val_df, test_df = make_group_splits(df)

    qwen_model, tokenizer, trainer = train_qwen(train_df, val_df)
    test_results = evaluate_group_test(
        qwen_model,
        tokenizer,
        train_df,
        val_df,
        test_df,
        rf_features=selected_features,
    )
    boundary_df = run_boundary_tests(qwen_model, tokenizer)

    demo = predict_pump_state_safeguarded(
        qwen_model,
        tokenizer,
        soil_moisture=18.0,
        water_level=70.0,
        rain_level=2.0,
    )
    print("\nDemo safeguarded prediction:", demo)

    readme = f"""Qwen2.5-1.5B QLoRA + Random Forest feature audit

Base model: {BASE_MODEL_PATH}
Input CSV: {DATA_PATH}
Rows: {len(df)}
Scenario episodes used for group splitting: {df['episode_group'].nunique()}
Qwen selected features: {selected_features}
Qwen split sizes: train={len(train_df)}, validation={len(val_df)}, test={len(test_df)}

Files:
- rf_groupcv_ablation.csv
- rf_grouped_feature_importance.csv
- rf_groupcv_oof_predictions.csv
- rf_groupcv_fold_metrics.csv
- source_rule_ablation.csv
- sensor_target_correlations.csv
- qwen_rf_group_test_predictions.csv
- qwen_rf_group_test_metrics.json
- qwen_boundary_tests.csv
- rf_pump_final.joblib
- qwen_pump_lora/ (QLoRA adapter and tokenizer)

Important: the dataset is synthetic and source-rule agreement should be audited.
For physical pump actuation, use a deterministic controller and hardware safety
interlocks as the final authority. Qwen predictions are diagnostic/experimental.
"""
    with open(WORK_DIR / "README.txt", "w", encoding="utf-8") as f:
        f.write(readme)

    adapter_zip = shutil.make_archive(
        "/kaggle/working/qwen_pump_lora_groupcv_adapter",
        "zip",
        str(QWEN_OUTPUT_DIR),
    )
    rf_zip = shutil.make_archive(
        "/kaggle/working/pump_rf_qwen_results",
        "zip",
        str(WORK_DIR),
    )
    print("\nCOMPLETE")
    print("Qwen adapter:", QWEN_OUTPUT_DIR)
    print("Qwen adapter ZIP:", adapter_zip)
    print("All RF/Qwen outputs ZIP:", rf_zip)


if __name__ == "__main__":
    main()
